# Del clic a la transacción

**Optativa II · M2-A1 · Aplicación, HTTP y persistencia**

Sara elige el asiento **B-7** y presiona **Confirmar reserva**.

La pregunta de este libro-laboratorio es:

> **¿Qué ocurre desde ese clic hasta que podemos demostrar qué quedó almacenado?**

No ejecutes todo de una vez. En cada bloque sigue esta secuencia:

**situación → pregunta → concepto → predicción → prueba → resultado → explicación → conexión con tu proyecto**

Cuando veas código, no necesitas memorizarlo completo. Debes poder explicar **qué problema está resolviendo y qué evidencia produce**.

## 1. Antes de HTTP: primero existió un problema de comunicación

A finales de los años ochenta, equipos y programas distintos necesitaban compartir información sin depender de que una persona copiara archivos manualmente de un lugar a otro.

En 1989 Tim Berners-Lee propuso la Web en CERN para facilitar ese intercambio. Hacia 1990 ya existían un navegador, un servidor y una primera forma de HTTP.

La necesidad era sencilla de formular:

> **Si un programa necesita algo que está en otro programa, ¿cómo lo pide y cómo recibe una respuesta de una forma que ambos entiendan?**

HTTP nació dentro de esa arquitectura como un protocolo de **solicitud y respuesta**.

**Importante:** HTTP no nació para controlar transacciones de bases de datos. Su trabajo principal es permitir la conversación entre programas.

<div style="max-width:900px;margin:18px auto 8px;">
<svg viewBox="0 0 900 220" width="100%" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Evolución conceptual de la comunicación web">
  <g fill="none" stroke="#191919" stroke-width="2" stroke-linecap="round">
    <path d="M80 110H820"/>
    <circle cx="120" cy="110" r="7"/><circle cx="350" cy="110" r="7"/><circle cx="580" cy="110" r="7"/><circle cx="800" cy="110" r="7"/>
  </g>
  <g fill="#191919" font-family="Arial,Helvetica,sans-serif" text-anchor="middle">
    <text x="120" y="62" font-size="24">1989</text><text x="120" y="145" font-size="16">necesidad</text><text x="120" y="168" font-size="12" fill="#666">compartir información</text>
    <text x="350" y="62" font-size="24">1990</text><text x="350" y="145" font-size="16">cliente + servidor</text><text x="350" y="168" font-size="12" fill="#666">pedir y devolver</text>
    <text x="580" y="62" font-size="24">HTTP crece</text><text x="580" y="145" font-size="16">más tipos de mensajes</text><text x="580" y="168" font-size="12" fill="#666">métodos, cabeceras, estados</text>
    <text x="800" y="62" font-size="24">Hoy</text><text x="800" y="145" font-size="16">APIs</text><text x="800" y="168" font-size="12" fill="#666">programas que colaboran</text>
  </g>
</svg>
</div>

### Qué debes notar

La tecnología aparece **después** de que existe una necesidad. No estudiaremos HTTP como una lista de comandos, sino como una respuesta a un problema de comunicación.

## 2. La idea mínima de HTTP

La primera conversación web era muy simple:

```text
cliente:  dame este recurso
servidor: aquí está
```

Una forma histórica muy sencilla podía verse así:

```text
GET /documento
```

Con el tiempo las aplicaciones necesitaron hacer más cosas: consultar información, enviar datos, crear operaciones, informar conflictos y comunicar fallos.

Por eso la conversación HTTP se hizo más expresiva.

### Una petición moderna necesita responder preguntas como estas

- **Método:** ¿qué tipo de acción estoy solicitando?
- **Ruta:** ¿sobre qué recurso quiero actuar?
- **Cabeceras:** ¿qué información ayuda a interpretar el mensaje?
- **Cuerpo:** ¿qué datos necesito enviar?

No memorices todavía los nombres. Primero entiende la necesidad que resuelve cada parte.

## 3. Volvemos al clic de Sara

Sara no escribe HTTP. Ella solo elige una opción y presiona un botón.

El recorrido es este:

**persona → interfaz → cliente → HTTP → servicio → base de datos → respuesta → persona**

- la **persona** inicia la acción;
- la **interfaz** recoge la información;
- el **cliente** construye la solicitud;
- **HTTP** transporta la solicitud y la respuesta;
- el **servicio** interpreta lo recibido y decide qué operación intentar;
- la **base de datos** aplica sus propias reglas.

### GET y POST aparecen porque las intenciones son diferentes

- `GET /reservas` → queremos **consultar** las reservas actuales;
- `POST /reservas` → queremos **enviar datos para que el servicio los procese**. En este laboratorio ese procesamiento crea una reserva.

> **Precisión:** POST no significa universalmente “crear”. Aquí lo usamos con ese propósito.

### ¿Y JSON?

Necesitamos una forma clara de escribir los datos que viajan:

```json
{
  "funcion": "F-01",
  "fila": "B",
  "puesto": 7,
  "persona": "Sara"
}
```

JSON es una **representación de datos**. HTTP es el **protocolo de comunicación**. No son la misma cosa.

## 4. La respuesta también necesita ser comprensible

Los códigos HTTP resumen el resultado de una petición. No los memorices aislados. Léelos siempre junto con **qué pediste, a qué ruta, qué mensaje recibiste y qué quedó en los datos**.

| Código | Qué te dice | En este laboratorio |
|---:|---|---|
| 200 | la operación solicitada fue atendida | consulta de reservas |
| 201 | se creó un recurso | reserva creada |
| 400 | hay un problema con los datos enviados | dato ausente o valor no admitido |
| 404 | no se encontró lo solicitado | función o ruta inexistente |
| 409 | la petición entra en conflicto con el estado actual | asiento ocupado |
| 415 | el formato recibido no es el esperado | se esperaba JSON |
| 500 | el servicio encontró un fallo interno | fallo provocado para estudiar rollback |

Ahora sí vamos a levantar un servicio HTTP real y observar estas respuestas.

In [ ]:
import json
import sqlite3
import tempfile
from pathlib import Path
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from threading import Thread
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

# Si vuelves a ejecutar desde arriba, cerramos una ejecución anterior.
if 'servidor' in globals():
    try:
        servidor.shutdown()
        servidor.server_close()
        hilo.join(timeout=3)
    except Exception:
        pass

if 'carpeta_temporal' in globals():
    try:
        carpeta_temporal.cleanup()
    except Exception:
        pass

carpeta_temporal = tempfile.TemporaryDirectory(prefix='ucc-cine-')
ruta_bd = Path(carpeta_temporal.name) / 'cine.sqlite'

def conectar():
    db = sqlite3.connect(ruta_bd, timeout=5)
    db.execute('PRAGMA foreign_keys = ON')
    return db

with conectar() as db:
    db.executescript("""
    CREATE TABLE funciones(
        id TEXT PRIMARY KEY,
        titulo TEXT NOT NULL
    );

    INSERT INTO funciones VALUES
        ('F-01', 'Viaje al futuro'),
        ('F-02', 'Ciudad nocturna');

    CREATE TABLE reservas(
        id INTEGER PRIMARY KEY,
        funcion TEXT NOT NULL REFERENCES funciones(id),
        fila TEXT NOT NULL CHECK(length(fila) = 1),
        puesto INTEGER NOT NULL CHECK(puesto > 0),
        persona TEXT NOT NULL,
        UNIQUE(funcion, fila, puesto)
    );

    CREATE TABLE bitacora(
        reserva_id INTEGER NOT NULL REFERENCES reservas(id),
        evento TEXT NOT NULL
    );
    """)

class ServicioReservas(BaseHTTPRequestHandler):
    def log_message(self, formato, *args):
        pass

    def responder(self, codigo, datos):
        cuerpo = json.dumps(datos, ensure_ascii=False).encode('utf-8')
        self.send_response(codigo)
        self.send_header('Content-Type', 'application/json; charset=utf-8')
        self.send_header('Content-Length', str(len(cuerpo)))
        self.end_headers()
        self.wfile.write(cuerpo)

    def do_GET(self):
        if self.path != '/reservas':
            return self.responder(404, {'mensaje': 'Ruta inexistente: usa /reservas'})

        with conectar() as db:
            filas = db.execute(
                'SELECT id, funcion, fila, puesto, persona FROM reservas ORDER BY id'
            ).fetchall()

        reservas = [
            {'id': r[0], 'funcion': r[1], 'fila': r[2], 'puesto': r[3], 'persona': r[4]}
            for r in filas
        ]
        self.responder(200, {'reservas': reservas})

    def do_POST(self):
        if self.path != '/reservas':
            return self.responder(404, {'mensaje': 'Ruta inexistente: usa /reservas'})

        content_type = self.headers.get('Content-Type', '')
        if not content_type.lower().startswith('application/json'):
            return self.responder(415, {'mensaje': 'Se esperaba Content-Type: application/json'})

        try:
            n = int(self.headers.get('Content-Length', '0'))
            if not 0 < n <= 4096:
                raise ValueError
            entrada = json.loads(self.rfile.read(n))
            if not isinstance(entrada, dict):
                raise ValueError

            funcion = entrada.get('funcion')
            fila = entrada.get('fila')
            puesto = entrada.get('puesto')
            persona = entrada.get('persona')

            if not isinstance(funcion, str) or not funcion.strip():
                raise ValueError
            if not isinstance(fila, str) or len(fila.strip()) != 1:
                raise ValueError
            if type(puesto) is not int or puesto <= 0:
                raise ValueError
            if not isinstance(persona, str) or not persona.strip():
                raise ValueError
        except (ValueError, UnicodeDecodeError, json.JSONDecodeError):
            return self.responder(400, {
                'mensaje': 'Revisa funcion, fila, puesto y persona; los datos no cumplen lo esperado'
            })

        db = conectar()
        try:
            db.execute('BEGIN IMMEDIATE')

            existe = db.execute(
                'SELECT 1 FROM funciones WHERE id = ?', (funcion,)
            ).fetchone()
            if not existe:
                db.rollback()
                return self.responder(404, {'mensaje': f'Función inexistente: {funcion}'})

            cur = db.execute(
                'INSERT INTO reservas(funcion, fila, puesto, persona) VALUES (?, ?, ?, ?)',
                (funcion, fila.upper(), puesto, persona.strip())
            )

            if self.headers.get('X-Ensayo-Fallo') == 'si':
                raise RuntimeError('Fallo didáctico antes de escribir la bitácora')

            db.execute(
                'INSERT INTO bitacora(reserva_id, evento) VALUES (?, ?)',
                (cur.lastrowid, 'reserva_creada')
            )
            identidad = cur.lastrowid
            db.commit()
            self.responder(201, {'mensaje': 'Reserva creada', 'id': identidad})

        except sqlite3.IntegrityError:
            db.rollback()
            self.responder(409, {'mensaje': 'Ese asiento ya está reservado para esa función'})
        except RuntimeError:
            db.rollback()
            self.responder(500, {
                'mensaje': 'Fallo provocado: la operación nueva se revirtió completa'
            })
        finally:
            db.close()

def iniciar_servicio():
    s = ThreadingHTTPServer(('127.0.0.1', 0), ServicioReservas)
    t = Thread(target=s.serve_forever, daemon=True)
    t.start()
    direccion = f'http://{s.server_address[0]}:{s.server_port}'
    return s, t, direccion

servidor, hilo, direccion = iniciar_servicio()

def solicitar(metodo, entrada=None, ruta='/reservas', content_type='application/json', fallo=False):
    datos = None if entrada is None else json.dumps(entrada).encode('utf-8')
    headers = {}
    if entrada is not None:
        headers['Content-Type'] = content_type
    if fallo:
        headers['X-Ensayo-Fallo'] = 'si'

    req = Request(direccion + ruta, data=datos, headers=headers, method=metodo)
    try:
        with urlopen(req, timeout=8) as respuesta:
            return respuesta.status, json.load(respuesta)
    except HTTPError as error:
        with error:
            return error.code, json.load(error)
    except URLError as error:
        return None, {'mensaje': f'No hubo respuesta HTTP: {error.reason}'}

print('Servicio HTTP local:', direccion)
print('Base temporal:', ruta_bd)

## 5. Primera pregunta observable: ¿qué hay antes de reservar?

Antes de crear nada, vamos a consultar.

### Predicción

Escribe mentalmente tu respuesta antes de ejecutar:

> **¿Qué debería devolver `GET /reservas` si todavía no hemos creado ninguna reserva?**

### Qué hace GET aquí

No crea ni modifica una reserva. Solo pide al servicio que nos muestre el estado actual que lee desde la base.

In [ ]:
print('GET /reservas antes de crear:')
print(solicitar('GET'))

## 6. Ahora sí: Sara intenta reservar B-7

El cliente enviará esta intención:

```text
POST /reservas
```

con estos datos:

```json
{
  "funcion": "F-01",
  "fila": "B",
  "puesto": 7,
  "persona": "Sara"
}
```

### Predicción

1. ¿Qué código esperas del POST?
2. ¿Cuántas reservas debería mostrar un GET inmediatamente después?
3. ¿Qué salida te permitiría afirmar que B-7 quedó realmente almacenado?

Recuerda: una respuesta `201` indica lo que informa el servicio. El GET posterior nos permite observar el estado que el servicio lee desde la base.

In [ ]:
sara = {
    'funcion': 'F-01',
    'fila': 'B',
    'puesto': 7,
    'persona': 'Sara'
}

print('POST de Sara:')
print(solicitar('POST', sara))

print('\nGET inmediatamente después:')
print(solicitar('GET'))

## 7. No todo rechazo significa lo mismo

Ahora vamos a enviar solicitudes que fallan por causas distintas.

### Antes de ejecutar, relaciona cada caso con una causa

- formato de contenido incorrecto;
- dato con forma o valor no admitido;
- función que no existe;
- asiento que ya está ocupado.

Aquí aparecen dos ideas diferentes:

### Validación del servicio

El servicio revisa si la entrada tiene la forma mínima esperada antes de intentar escribir.

### Integridad de la base

La base conserva restricciones que no deben romperse, incluso si otro proceso intenta escribir por otra vía.

Una misma regla puede aparecer en ambas capas por razones distintas: **responder con claridad** y **proteger los datos**.

In [ ]:
casos = [
    ('Content-Type incorrecto',
     {'funcion':'F-01','fila':'C','puesto':2,'persona':'Marta'},
     'text/plain'),
    ('puesto = 0',
     {'funcion':'F-01','fila':'C','puesto':0,'persona':'Marta'},
     'application/json'),
    ('función inexistente',
     {'funcion':'F-99','fila':'C','puesto':2,'persona':'Marta'},
     'application/json'),
    ('B-7 duplicado',
     {'funcion':'F-01','fila':'B','puesto':7,'persona':'Luis'},
     'application/json'),
]

for nombre, entrada, tipo in casos:
    print(f'{nombre:24}', solicitar('POST', entrada, content_type=tipo))

print('\nEstado después de todos los rechazos:')
print(solicitar('GET'))

## 8. Concurrencia: dos personas ven el mismo asiento libre

Ahora aparece un problema que una prueba secuencial no muestra bien.

Ana y Luis ven **D-4** disponible y presionan Reservar casi al mismo tiempo.

Las dos solicitudes pueden estar bien construidas. El conflicto aparece porque ambas quieren producir el mismo estado final.

### Pregunta

> **¿Qué evita que terminen existiendo dos filas para D-4?**

HTTP transporta ambas solicitudes. La protección decisiva está en la base:

```sql
UNIQUE(funcion, fila, puesto)
```

### Predicción

Esperamos observar:

- un `201`;
- un `409`;
- una sola reserva D-4.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from threading import Barrier

barrera = Barrier(2)

def competir(persona):
    entrada = {
        'funcion': 'F-01',
        'fila': 'D',
        'puesto': 4,
        'persona': persona
    }
    barrera.wait()
    return persona, solicitar('POST', entrada)

with ThreadPoolExecutor(max_workers=2) as pool:
    resultados = list(pool.map(competir, ['Ana', 'Luis']))

for resultado in resultados:
    print(resultado)

estado = solicitar('GET')
reservas_d4 = [r for r in estado[1]['reservas'] if r['fila'] == 'D' and r['puesto'] == 4]

print('\nReservas D-4 almacenadas:', reservas_d4)

codigos = sorted(r[1][0] for r in resultados)
assert codigos == [201, 409]
assert len(reservas_d4) == 1

## 9. Transacción: una operación puede tener varios cambios

En nuestro servicio, crear una reserva hace dos escrituras:

1. insertar la reserva;
2. registrar en `bitacora` que la reserva fue creada.

¿Qué pasaría si la primera escritura funcionara y la segunda fallara?

Tendríamos una operación a medias.

Una **transacción de base de datos** trata ambos cambios como una sola unidad lógica. Si no puede terminar, los cambios nuevos se deshacen mediante un **rollback**.

### No confundas

- HTTP lleva la petición desde el cliente al servicio.
- La transacción protege las escrituras dentro de la base.

### Predicción

Provocaremos un fallo al intentar reservar **E-2** después de insertar la reserva y antes de escribir la bitácora.

Después del `500`, ¿debería existir E-2?

In [ ]:
fallida = {
    'funcion': 'F-01',
    'fila': 'E',
    'puesto': 2,
    'persona': 'Nora'
}

print('POST con fallo provocado:')
print(solicitar('POST', fallida, fallo=True))

print('\nEstado posterior:')
estado = solicitar('GET')
print(estado)

with conectar() as db:
    reservas = db.execute('SELECT COUNT(*) FROM reservas').fetchone()[0]
    bitacora = db.execute('SELECT COUNT(*) FROM bitacora').fetchone()[0]
    e2 = db.execute(
        "SELECT COUNT(*) FROM reservas WHERE funcion='F-01' AND fila='E' AND puesto=2"
    ).fetchone()[0]

print('Conteo reservas:', reservas)
print('Conteo bitácora:', bitacora)
print('Reservas E-2:', e2)

assert e2 == 0
assert reservas == bitacora

## 10. Persistencia: ¿qué pasa si el servicio se reinicia?

Hasta ahora el servicio está ejecutándose como un proceso en memoria. Pero la base SQLite está en un archivo.

Vamos a:

1. detener el servicio;
2. conservar el archivo SQLite;
3. iniciar un nuevo servicio;
4. volver a hacer GET.

### Predicción

Si los datos persisten fuera del proceso del servicio, las reservas anteriores deberían seguir apareciendo.

Esto **no** demuestra respaldo. Si elimináramos el archivo SQLite, este laboratorio no tendría otra copia desde la cual recuperar los datos.

In [ ]:
servidor.shutdown()
servidor.server_close()
hilo.join(timeout=3)

servidor, hilo, direccion = iniciar_servicio()

print('Servicio reiniciado:', direccion)
print('GET después del reinicio:')
print(solicitar('GET'))

## 11. HTTP es una opción de ingeniería, no la única

HTTP encaja muy bien cuando un cliente hace una solicitud y espera una respuesta. Pero otras necesidades pueden llevar a otras tecnologías.

| Necesidad | Tecnología que podrías investigar |
|---|---|
| solicitud/respuesta web | HTTP |
| comunicación bidireccional persistente | WebSocket |
| mensajes pequeños y frecuentes en IoT | MQTT |
| llamadas estructuradas entre servicios | gRPC |
| trabajo asíncrono | colas o brokers |

No necesitas aprenderlas hoy. La idea importante es esta:

> **Primero identifica cómo necesitan comunicarse tus componentes. Después eliges el mecanismo.**

## 12. Ahora llévalo a tu proyecto

Elige **una operación concreta** de tu proyecto. No copies el caso del cine. Usa tu propio problema.

Completa estas preguntas:

| Pregunta | Tu proyecto |
|---|---|
| ¿Quién inicia la acción? | |
| ¿Qué acción realiza? | |
| ¿Qué datos necesita comunicar el cliente? | |
| ¿Qué método y ruta podrían representar la operación? | |
| ¿Qué debe validar el servicio? | |
| ¿Qué reglas deben permanecer en la base? | |
| ¿Qué conflicto puede ocurrir si dos operaciones llegan casi al mismo tiempo? | |
| ¿Hay varios cambios que deban completarse juntos? | |
| ¿Cómo demostrarás el estado final? | |

### Hacia E2

Estas respuestas pueden convertirse en el diseño y la prueba de una operación real de tu proyecto.

### Hacia E3

Cualquier capacidad analítica o inteligente tendrá una base más sólida si puedes explicar de dónde llegan los datos, qué reglas los protegen y cómo compruebas su estado.

## 13. Tu experimento

Cambia **una sola condición** y escribe primero una predicción.

Ejemplos:

- reserva otro asiento;
- cambia de función;
- omite un campo;
- usa una fila con dos letras;
- intenta otra duplicidad.

Escribe algo como:

> “Espero código ___ y espero que después existan ___ reservas porque ___”.

Después ejecuta tu prueba y compara la evidencia con lo que predijiste.

In [ ]:
# TU EXPERIMENTO
# 1. Escribe tu predicción en un comentario.
# 2. Cambia una sola condición.
# 3. Ejecuta POST o GET.
# 4. Consulta el estado después.
#
# Ejemplo:
# entrada = {'funcion':'F-02','fila':'A','puesto':1,'persona':'Tu nombre'}
# print('POST:', solicitar('POST', entrada))
# print('GET :', solicitar('GET'))

## 14. Cierre: explica el recorrido sin mirar el código

Intenta responder con tus propias palabras:

1. ¿Quién inicia la acción y quién envía la solicitud HTTP?
2. ¿Por qué históricamente hizo falta un protocolo como HTTP?
3. ¿Qué diferencia hay entre GET y POST en este laboratorio?
4. ¿Qué diferencia hay entre HTTP y JSON?
5. ¿Por qué un `201` y un GET posterior son evidencias diferentes?
6. ¿Qué diferencia hay entre validar la entrada y proteger la integridad de la base?
7. ¿Qué evita que Ana y Luis terminen ambos con D-4?
8. ¿Qué problema resuelve una transacción?
9. ¿Por qué persistencia no significa respaldo?
10. ¿Qué parte de este recorrido puedes aplicar a tu proyecto?

Si puedes responderlas con ejemplos, ya no estás memorizando términos: estás entendiendo el sistema.

In [ ]:
# Cierra los recursos creados por la práctica.
try:
    servidor.shutdown()
    servidor.server_close()
    hilo.join(timeout=3)
except Exception:
    pass

try:
    carpeta_temporal.cleanup()
except Exception:
    pass

print('Laboratorio cerrado: servicio detenido y base temporal eliminada.')